# Teaching_Video_Generation（Colab CPU + Drive 输出）

**上传位置：** 把本文件放到 Drive 的

`MyDrive/education_video/run_pipeline.ipynb`

（即与 `api_usage_tracker.py`、`eval.ipynb` 同级的 `education_video` 根目录）

| 东西 | 位置 |
|---|---|
| 本 notebook | `MyDrive/education_video/run_pipeline.ipynb` |
| 代码仓库 | Colab 里 `git clone`，不要上传 `Teaching_Video_Generation` |
| 运行输出 | `MyDrive/education_video/teachgen_runs/` |

Cell 3 必须打印 `DEPS OK` 再跑后面。


In [5]:
# 1) Mount Drive + OpenAI key

from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
from getpass import getpass
from pathlib import Path

from google.colab import drive  # type: ignore

drive.mount("/content/drive", force_remount=False)

REPO = Path("/content/Teaching_Video_Generation")
REPO_URL = "https://github.com/Hyan-Yao/Teaching_Video_Generation.git"
REPO_BRANCH = "evaluator"

DRIVE_ROOT = Path("/content/drive/MyDrive/education_video")
RUN_DIR = DRIVE_ROOT / "teachgen_runs"
REQUEST_PATH = DRIVE_ROOT / "requests" / "web_demo_regression.json"
RUN_DIR.mkdir(parents=True, exist_ok=True)
REQUEST_PATH.parent.mkdir(parents=True, exist_ok=True)

MAX_WORKERS = 20
TEXT_MODEL = "gpt-5"

if not os.environ.get("OPENAI_API_KEY"):
    try:
        from google.colab import userdata  # type: ignore
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except Exception:
        pass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ").strip()
if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is required")

os.environ["OPENAI_BASE_URL"] = "https://api.openai.com/v1"
os.environ["TEACHGEN_EVAL_BACKEND"] = "openai"
os.environ["TEACHGEN_EVAL_TEXT_MODEL"] = TEXT_MODEL
os.environ["TEACHGEN_EVAL_VIDEO_MODEL"] = TEXT_MODEL
os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ.setdefault("LANG", "C.UTF-8")
os.environ.setdefault("LC_ALL", "C.UTF-8")

print("REPO:", REPO)
print("RUN_DIR:", RUN_DIR)
print("python:", sys.executable)
print("OPENAI set:", bool(os.environ.get("OPENAI_API_KEY")))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
REPO: /content/Teaching_Video_Generation
RUN_DIR: /content/drive/MyDrive/education_video/teachgen_runs
python: /usr/bin/python3
OPENAI set: True


In [6]:
# 2) Clone evaluator branch onto Colab disk

if (REPO / "teachgen" / "pipeline.py").is_file():
    print("Repo already present:", REPO)
else:
    if REPO.exists():
        shutil.rmtree(REPO)
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, str(REPO)])
    print("Cloned", REPO_BRANCH, "->", REPO)

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("HEAD:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())


Repo already present: /content/Teaching_Video_Generation
HEAD: 82d4124


In [7]:
# 3) Install deps — manim is REQUIRED
# Do NOT import numpy/scipy/manim in this Colab kernel after reinstalling them.
# Verify only via a fresh subprocess (same interpreter teachgen will use).

print("Using python:", sys.executable)
subprocess.check_call(["apt-get", "-qq", "update"])
subprocess.check_call([
    "apt-get", "-qq", "install", "-y",
    "ffmpeg", "git",
    "libcairo2-dev", "libpango1.0-dev", "libffi-dev",
    "pkg-config", "libcairo2", "libpangocairo-1.0-0",
])

def pip_install(*args: str) -> None:
    cmd = [sys.executable, "-m", "pip", "install", "-q", *args]
    print("+", " ".join(cmd), flush=True)
    subprocess.check_call(cmd)

def pip_uninstall(*pkgs: str) -> None:
    cmd = [sys.executable, "-m", "pip", "uninstall", "-y", *pkgs]
    print("+", " ".join(cmd), flush=True)
    subprocess.call(cmd)

pip_install("-U", "pip", "setuptools", "wheel")
pip_install("-r", str(REPO / "requirements.txt"))
pip_install("openai", "python-dotenv")

# The full code2video pin set corrupts Colab's NumPy/SciPy environment.
print("Skipping full code2video/requirements.txt on Colab (avoids numpy breakage)")

pip_uninstall("numpy", "scipy", "manim", "ManimPango")
pip_install(
    "--no-cache-dir",
    "numpy==2.1.3",
    "scipy==1.14.1",
    "manim==0.19.0",
    "ManimPango==0.6.0",
    "mapbox-earcut",
    "moderngl",
    "moderngl-window",
    "skia-pathops",
    "isosurfaces",
    "networkx",
    "decorator",
    "watchdog",
    "tqdm",
    "rich",
    "pydub",
    "av",
)

manim_cli = shutil.which("manim")
if not manim_cli:
    candidate = Path(sys.executable).resolve().parent / "manim"
    if candidate.is_file():
        os.environ["PATH"] = str(candidate.parent) + os.pathsep + os.environ.get("PATH", "")
        manim_cli = shutil.which("manim")
print("manim CLI:", manim_cli)
if not manim_cli:
    raise RuntimeError("manim CLI not on PATH after install")

# Fresh-process check only; importing into this kernel can reuse stale NumPy modules.
chk = subprocess.check_output(
    [
        sys.executable,
        "-c",
        "import numpy, scipy, manim; print(numpy.__version__, scipy.__version__, getattr(manim,'__version__','ok'))",
    ],
    text=True,
).strip()
print("subprocess check:", chk)
print("DEPS OK")


Using python: /usr/bin/python3
+ /usr/bin/python3 -m pip install -q -U pip setuptools wheel
+ /usr/bin/python3 -m pip install -q -r /content/Teaching_Video_Generation/requirements.txt
+ /usr/bin/python3 -m pip install -q openai python-dotenv
Skipping full code2video/requirements.txt on Colab (avoids numpy breakage)
+ /usr/bin/python3 -m pip uninstall -y numpy scipy manim ManimPango
+ /usr/bin/python3 -m pip install -q --no-cache-dir numpy==2.1.3 scipy==1.14.1 manim==0.19.0 ManimPango==0.6.0 mapbox-earcut moderngl moderngl-window skia-pathops isosurfaces networkx decorator watchdog tqdm rich pydub av
manim CLI: /usr/local/bin/manim
subprocess check: 2.1.3 1.14.1 0.19.0
DEPS OK


In [8]:
# 4) Patch evaluator for official OpenAI

(REPO / "teachgen" / "eval" / "text_llm.py").write_text(json.loads('"import os\\nimport hashlib\\nfrom typing import TypeVar\\n\\nfrom openai import OpenAI\\nfrom pydantic import BaseModel, ValidationError\\n\\nOutputModel = TypeVar(\\"OutputModel\\", bound=BaseModel)\\n\\nGPT5_MIN_COMPLETION_TOKENS = 12000\\n\\n\\nclass TextLLM:\\n    def __init__(self, model: str | None = None):\\n        backend, base_url, api_key = _resolve_backend()\\n        default_model = \\"gpt-5\\" if backend == \\"openai\\" else \\"openai/gpt-4.1\\"\\n        self.backend = backend\\n        self.model = model or os.environ.get(\\"TEACHGEN_EVAL_TEXT_MODEL\\", default_model)\\n        self.max_retries = int(os.environ.get(\\"TEACHGEN_EVAL_TEXT_RETRIES\\", \\"3\\"))\\n        self.client = OpenAI(base_url=base_url, api_key=api_key)\\n\\n    def analyze(\\n        self,\\n        prompt: str,\\n        output_model: type[OutputModel],\\n    ) -> OutputModel:\\n        last_error: Exception | None = None\\n        last_answer = \\"\\"\\n\\n        for attempt in range(self.max_retries + 1):\\n            kwargs = {\\n                \\"model\\": self.model,\\n                \\"messages\\": [{\\"role\\": \\"user\\", \\"content\\": prompt}],\\n                \\"temperature\\": 0,\\n                \\"seed\\": 12345 + attempt,\\n                \\"max_tokens\\": 8000,\\n                \\"response_format\\": {\\n                    \\"type\\": \\"json_schema\\",\\n                    \\"json_schema\\": {\\n                        \\"name\\": output_model.__name__,\\n                        \\"strict\\": True,\\n                        \\"schema\\": output_model.model_json_schema(),\\n                    },\\n                },\\n            }\\n            if self.backend == \\"openrouter\\":\\n                kwargs[\\"extra_body\\"] = {\\n                    \\"session_id\\": _stable_session_id(\\n                        self.model,\\n                        output_model.__name__,\\n                        prompt,\\n                        attempt=attempt,\\n                    ),\\n                }\\n            kwargs = _normalize_chat_kwargs(kwargs)\\n\\n            response = self.client.chat.completions.create(**kwargs)\\n\\n            answer = _message_content(response)\\n            last_answer = answer or \\"\\"\\n            if not answer:\\n                last_error = RuntimeError(f\\"Model {self.model} returned no content\\")\\n                continue\\n\\n            try:\\n                return output_model.model_validate_json(answer)\\n            except ValidationError as exc:\\n                last_error = exc\\n\\n        snippet = last_answer[:500].replace(\\"\\\\n\\", \\"\\\\\\\\n\\")\\n        raise RuntimeError(\\n            f\\"Model {self.model} did not return valid JSON after \\"\\n            f\\"{self.max_retries + 1} attempts for {output_model.__name__}. \\"\\n            f\\"Last error: {last_error}. Last response prefix: {snippet}\\"\\n        )\\n\\n\\ndef _resolve_backend() -> tuple[str, str, str]:\\n    \\"\\"\\"Prefer official OpenAI when OPENAI_API_KEY is set; else OpenRouter.\\"\\"\\"\\n    forced = os.environ.get(\\"TEACHGEN_EVAL_BACKEND\\", \\"\\").strip().lower()\\n    openai_key = os.environ.get(\\"OPENAI_API_KEY\\", \\"\\").strip()\\n    openrouter_key = os.environ.get(\\"OPENROUTER_API_KEY\\", \\"\\").strip()\\n\\n    if forced == \\"openai\\" or (forced != \\"openrouter\\" and openai_key):\\n        if not openai_key:\\n            raise RuntimeError(\\"OPENAI_API_KEY is required for TEACHGEN_EVAL_BACKEND=openai\\")\\n        return (\\n            \\"openai\\",\\n            os.environ.get(\\"OPENAI_BASE_URL\\", \\"https://api.openai.com/v1\\"),\\n            openai_key,\\n        )\\n\\n    if not openrouter_key:\\n        raise RuntimeError(\\n            \\"Set OPENAI_API_KEY (preferred) or OPENROUTER_API_KEY for the evaluator.\\"\\n        )\\n    return \\"openrouter\\", \\"https://openrouter.ai/api/v1\\", openrouter_key\\n\\n\\ndef _normalize_chat_kwargs(kwargs: dict) -> dict:\\n    normalized = dict(kwargs)\\n    model = str(normalized.get(\\"model\\", \\"\\")).casefold()\\n    if not (model.startswith(\\"gpt-5\\") or \\"/gpt-5\\" in model):\\n        return normalized\\n\\n    if \\"max_tokens\\" in normalized:\\n        requested = normalized.pop(\\"max_tokens\\") or 0\\n        normalized[\\"max_completion_tokens\\"] = max(int(requested), GPT5_MIN_COMPLETION_TOKENS)\\n    normalized.pop(\\"temperature\\", None)\\n    normalized.pop(\\"seed\\", None)\\n    return normalized\\n\\n\\ndef _message_content(response) -> str:\\n    try:\\n        content = response.choices[0].message.content\\n    except Exception:\\n        return \\"\\"\\n    if content is None:\\n        return \\"\\"\\n    if isinstance(content, str):\\n        return content\\n    if isinstance(content, list):\\n        parts = []\\n        for item in content:\\n            if isinstance(item, dict):\\n                text = item.get(\\"text\\")\\n            else:\\n                text = getattr(item, \\"text\\", None)\\n            if text:\\n                parts.append(text)\\n        return \\"\\\\n\\".join(parts)\\n    return str(content)\\n\\n\\ndef _stable_session_id(model: str, schema_name: str, prompt: str, *, attempt: int = 0) -> str:\\n    key = f\\"{model}\\\\n{schema_name}\\\\n{attempt}\\\\n{prompt}\\".encode(\\"utf-8\\")\\n    return f\\"eval-{hashlib.sha256(key).hexdigest()[:32]}\\"\\n"'), encoding="utf-8")
(REPO / "teachgen" / "eval" / "video_llm.py").write_text(json.loads('"import base64\\nimport os\\nimport hashlib\\nfrom io import BytesIO\\nfrom pathlib import Path\\nfrom openai import OpenAI\\nfrom typing import TypeVar\\nfrom pydantic import BaseModel, ValidationError\\n\\nfrom teachgen.eval.text_llm import (\\n    GPT5_MIN_COMPLETION_TOKENS,\\n    _message_content,\\n    _normalize_chat_kwargs,\\n    _resolve_backend,\\n)\\n\\nOutputModel = TypeVar(\\"OutputModel\\", bound=BaseModel)\\n\\n\\nclass VideoLLM:\\n    def __init__(self, model: str | None = None):\\n        backend, base_url, api_key = _resolve_backend()\\n        default_model = (\\n            \\"gpt-5\\" if backend == \\"openai\\" else \\"google/gemini-3.5-flash\\"\\n        )\\n        self.backend = backend\\n        self.model = model or os.environ.get(\\"TEACHGEN_EVAL_VIDEO_MODEL\\", default_model)\\n        self.max_retries = int(os.environ.get(\\"TEACHGEN_EVAL_VIDEO_RETRIES\\", \\"3\\"))\\n        self.num_frames = int(os.environ.get(\\"TEACHGEN_EVAL_VIDEO_FRAMES\\", \\"16\\"))\\n        self.client = OpenAI(base_url=base_url, api_key=api_key)\\n\\n    def analyze(\\n        self,\\n        video_path: str | Path,\\n        prompt: str,\\n        output_model: type[OutputModel],\\n    ) -> OutputModel:\\n        path = Path(video_path)\\n        if not path.is_file():\\n            raise FileNotFoundError(f\\"Video not found: {path}\\")\\n\\n        if self.backend == \\"openai\\":\\n            return self._analyze_openai_frames(path, prompt, output_model)\\n        return self._analyze_openrouter_video(path, prompt, output_model)\\n\\n    def _analyze_openai_frames(\\n        self,\\n        path: Path,\\n        prompt: str,\\n        output_model: type[OutputModel],\\n    ) -> OutputModel:\\n        frames = _sample_frames(path, self.num_frames)\\n        content = [\\n            {\\n                \\"type\\": \\"text\\",\\n                \\"text\\": (\\n                    f\\"{prompt}\\\\n\\\\n\\"\\n                    f\\"You are shown {len(frames)} frames sampled in chronological order \\"\\n                    \\"from this video chunk. Infer motion, pacing, and continuity from the \\"\\n                    \\"ordered frames. Timestamps in evidence should still refer to the \\"\\n                    \\"chunk-local timeline described above.\\"\\n                ),\\n            }\\n        ]\\n        for img in frames:\\n            b64 = base64.b64encode(img).decode(\\"utf-8\\")\\n            content.append(\\n                {\\n                    \\"type\\": \\"image_url\\",\\n                    \\"image_url\\": {\\"url\\": f\\"data:image/png;base64,{b64}\\"},\\n                }\\n            )\\n\\n        last_error: Exception | None = None\\n        last_answer = \\"\\"\\n        for attempt in range(self.max_retries + 1):\\n            kwargs = {\\n                \\"model\\": self.model,\\n                \\"messages\\": [{\\"role\\": \\"user\\", \\"content\\": content}],\\n                \\"temperature\\": 0,\\n                \\"seed\\": 12345 + attempt,\\n                \\"max_tokens\\": 8000,\\n                \\"response_format\\": {\\n                    \\"type\\": \\"json_schema\\",\\n                    \\"json_schema\\": {\\n                        \\"name\\": output_model.__name__,\\n                        \\"strict\\": True,\\n                        \\"schema\\": output_model.model_json_schema(),\\n                    },\\n                },\\n            }\\n            kwargs = _normalize_chat_kwargs(kwargs)\\n            # Ensure GPT-5 still gets enough completion budget for long schemas.\\n            if \\"max_completion_tokens\\" in kwargs:\\n                kwargs[\\"max_completion_tokens\\"] = max(\\n                    int(kwargs[\\"max_completion_tokens\\"]),\\n                    GPT5_MIN_COMPLETION_TOKENS,\\n                )\\n\\n            response = self.client.chat.completions.create(**kwargs)\\n            answer = _message_content(response)\\n            last_answer = answer or \\"\\"\\n            if not answer:\\n                last_error = RuntimeError(\\n                    f\\"Model {self.model} returned no content for video {path}\\"\\n                )\\n                continue\\n            try:\\n                return output_model.model_validate_json(answer)\\n            except ValidationError as exc:\\n                last_error = exc\\n\\n        snippet = last_answer[:500].replace(\\"\\\\n\\", \\"\\\\\\\\n\\")\\n        raise RuntimeError(\\n            f\\"Model {self.model} did not return valid JSON after \\"\\n            f\\"{self.max_retries + 1} attempts for {output_model.__name__}. \\"\\n            f\\"Last error: {last_error}. Last response prefix: {snippet}\\"\\n        )\\n\\n    def _analyze_openrouter_video(\\n        self,\\n        path: Path,\\n        prompt: str,\\n        output_model: type[OutputModel],\\n    ) -> OutputModel:\\n        encoded_video = base64.b64encode(path.read_bytes()).decode(\\"utf-8\\")\\n        video_url = f\\"data:video/mp4;base64,{encoded_video}\\"\\n\\n        response = self.client.chat.completions.create(\\n            model=self.model,\\n            messages=[\\n                {\\n                    \\"role\\": \\"user\\",\\n                    \\"content\\": [\\n                        {\\"type\\": \\"text\\", \\"text\\": prompt},\\n                        {\\n                            \\"type\\": \\"video_url\\",\\n                            \\"video_url\\": {\\"url\\": video_url},\\n                        },\\n                    ],\\n                }\\n            ],\\n            temperature=0,\\n            seed=12345,\\n            response_format={\\n                \\"type\\": \\"json_schema\\",\\n                \\"json_schema\\": {\\n                    \\"name\\": output_model.__name__,\\n                    \\"strict\\": True,\\n                    \\"schema\\": output_model.model_json_schema(),\\n                },\\n            },\\n            extra_body={\\n                \\"session_id\\": _stable_session_id(\\n                    self.model, output_model.__name__, prompt\\n                ),\\n            },\\n        )\\n        answer = _message_content(response)\\n        if not answer:\\n            raise RuntimeError(f\\"Model {self.model} returned no content for video {path}\\")\\n\\n        return output_model.model_validate_json(answer)\\n\\n\\ndef _sample_frames(video_path: Path, n: int) -> list[bytes]:\\n    from teachgen.mpcompat import VideoFileClip\\n    from PIL import Image\\n\\n    clip = VideoFileClip(str(video_path))\\n    try:\\n        duration = float(clip.duration or 0.0)\\n        if duration <= 0:\\n            raise RuntimeError(f\\"Video has invalid duration: {video_path}\\")\\n        out: list[bytes] = []\\n        for i in range(max(1, n)):\\n            t = duration * (i + 0.5) / max(1, n)\\n            frame = clip.get_frame(min(t, max(0.0, duration - 0.05)))\\n            buf = BytesIO()\\n            Image.fromarray(frame).save(buf, format=\\"PNG\\")\\n            out.append(buf.getvalue())\\n        return out\\n    finally:\\n        clip.close()\\n\\n\\ndef _stable_session_id(model: str, schema_name: str, prompt: str) -> str:\\n    key = f\\"{model}\\\\n{schema_name}\\\\n{prompt}\\".encode(\\"utf-8\\")\\n    return f\\"eval-{hashlib.sha256(key).hexdigest()[:32]}\\"\\n"'), encoding="utf-8")
print("Patched teachgen/eval/{text_llm,video_llm}.py")


Patched teachgen/eval/{text_llm,video_llm}.py


In [ ]:
# 4b) Patch Code2Video reliability for Colab
# Keeps overall asset concurrency at 20 while bounding CPU-heavy Manim renders to 2.


def replace_once(path: Path, old: str, new: str) -> None:
    text = path.read_text(encoding="utf-8")
    if old not in text:
        raise RuntimeError(f"Patch anchor not found in {path}: {old[:100]!r}")
    path.write_text(text.replace(old, new, 1), encoding="utf-8")


scope_path = REPO / "code2video" / "scope_refine.py"
agent_path = REPO / "code2video" / "agent.py"
animation_path = REPO / "teachgen" / "renderers" / "animation.py"

# Deterministically rename the generated TeachingScene subclass to SegNScene.
replace_once(scope_path, "import re\n", "import ast\nimport re\nimport sys\n")
scene_normalizer = '''\n\ndef normalize_scene_class_name(code: str, section_id: str) -> str:\n    expected = f"{section_id.title().replace('_', '')}Scene"\n    try:\n        tree = ast.parse(code)\n    except SyntaxError:\n        return code\n    classes = [node for node in ast.walk(tree) if isinstance(node, ast.ClassDef)]\n    if any(node.name == expected for node in classes):\n        return code\n\n    def base_names(node):\n        names = set()\n        for base in node.bases:\n            if isinstance(base, ast.Name):\n                names.add(base.id)\n            elif isinstance(base, ast.Attribute):\n                names.add(base.attr)\n        return names\n\n    candidates = [\n        node for node in classes\n        if node.name != "TeachingScene" and "TeachingScene" in base_names(node)\n    ]\n    if len(candidates) != 1:\n        candidates = [\n            node for node in classes\n            if node.name != "TeachingScene" and "Scene" in base_names(node)\n        ]\n    if len(candidates) != 1:\n        return code\n    candidate = candidates[0]\n    lines = code.splitlines(keepends=True)\n    index = candidate.lineno - 1\n    lines[index] = re.sub(\n        rf"(\\bclass\\s+){re.escape(candidate.name)}(\\s*\\()",\n        rf"\\g<1>{expected}\\g<2>",\n        lines[index],\n        count=1,\n    )\n    return "".join(lines)\n'''
replace_once(scope_path, "logger = logging.getLogger(__name__)\n", "logger = logging.getLogger(__name__)\n" + scene_normalizer)
replace_once(
    scope_path,
    '        test_file = output_dir / f"test_{section_id}.py"\n',
    '        test_file = output_dir / f"test_{section_id}.py"\n        code = normalize_scene_class_name(code, section_id)\n',
)
replace_once(scope_path, '            cmd = ["python", "-c",', '            cmd = [sys.executable, "-c",')
replace_once(
    scope_path,
    "            5. Use proper Manim CE v0.19.0 syntax\n",
    "            5. Use proper Manim CE v0.19.0 syntax\n            6. Preserve exactly one scene class named {section_id.title().replace('_', '')}Scene\n",
)

# Normalize both freshly generated and repaired code; allow 360s per CPU render.
replace_once(
    agent_path,
    "        code = replace_base_class(code, base_class)\n",
    "        code = replace_base_class(code, base_class)\n        code = normalize_scene_class_name(code, section.id)\n",
)
replace_once(
    agent_path,
    '                result = subprocess.run(cmd, capture_output=True, text=True, cwd=self.output_dir, timeout=180)\n',
    '                timeout_seconds = int(os.environ.get("TEACHGEN_MANIM_TIMEOUT_SECONDS", "360"))\n                result = subprocess.run(\n                    cmd, capture_output=True, text=True, cwd=self.output_dir,\n                    timeout=timeout_seconds,\n                )\n',
)
replace_once(
    agent_path,
    "                if fixed_code:\n                    self.section_codes[section_id] = fixed_code\n",
    "                if fixed_code:\n                    fixed_code = normalize_scene_class_name(fixed_code, section_id)\n                    self.section_codes[section_id] = fixed_code\n",
)

# Load Code2Video once and keep only two render_section calls active.
replace_once(animation_path, "import os\n", "import os\nimport threading\n")
replace_once(
    animation_path,
    "# Code2Video renders at -ql (854x480 @ 15fps); the compositor upscales to 1080p.\n",
    "# Code2Video renders at -ql (854x480 @ 15fps); the compositor upscales to 1080p.\n_CODE2VIDEO_LOAD_LOCK = threading.Lock()\n_CODE2VIDEO_CACHE = None\n_MANIM_RENDER_SEMAPHORE = threading.BoundedSemaphore(\n    max(1, int(os.environ.get(\"TEACHGEN_MANIM_MAX_CONCURRENCY\", \"2\")))\n)\n",
)
replace_once(animation_path, "            max_fix_bug_tries=3,\n", "            max_fix_bug_tries=2,\n")
replace_once(
    animation_path,
    "        ok = agent.render_section(section)\n",
    "        with _MANIM_RENDER_SEMAPHORE:\n            ok = agent.render_section(section)\n",
)

animation_text = animation_path.read_text(encoding="utf-8")
start = animation_text.index("def _load_code2video(repo_root: Path):")
end = animation_text.index("\ndef _duration(path: Path)", start)
new_loader = '''def _load_code2video(repo_root: Path):\n    global _CODE2VIDEO_CACHE\n    if _CODE2VIDEO_CACHE is not None:\n        return _CODE2VIDEO_CACHE\n    with _CODE2VIDEO_LOAD_LOCK:\n        if _CODE2VIDEO_CACHE is not None:\n            return _CODE2VIDEO_CACHE\n        code2video_dir = repo_root / "code2video"\n        for p in (str(code2video_dir), str(repo_root)):\n            if p not in sys.path:\n                sys.path.insert(0, p)\n        agent_path = code2video_dir / "agent.py"\n        spec = importlib.util.spec_from_file_location("teachgen_code2video_agent", agent_path)\n        if spec is None or spec.loader is None:\n            raise ImportError(f"Could not load code2video agent from {agent_path}")\n        agent = importlib.util.module_from_spec(spec)\n        sys.modules["teachgen_code2video_agent"] = agent\n        spec.loader.exec_module(agent)\n        from prompts import base_class\n        _CODE2VIDEO_CACHE = (agent, base_class)\n        return _CODE2VIDEO_CACHE\n'''
animation_path.write_text(animation_text[:start] + new_loader + animation_text[end:], encoding="utf-8")

os.environ["TEACHGEN_MANIM_MAX_CONCURRENCY"] = "2"
os.environ["TEACHGEN_MANIM_TIMEOUT_SECONDS"] = "360"
print("Code2Video patch applied: total workers=20, Manim=2, timeout=360s, deterministic Scene names")

In [9]:
# 5) Write request JSON to Drive

REQUEST = {
    "request_id": "web_demo_regression",
    "course": {
        "topic": "Regression for explanation vs. regression for prediction",
        "learning_goal": (
            "Deliver a clear lecture explaining the differences between using regression "
            "for explanation and using regression for prediction. Explain that the purpose "
            "of explanation is to identify and interpret important influencing factors, "
            "while the purpose of prediction is to build a model that can make accurate "
            "predictions on new data."
        ),
        "key_learning_points": [
            "Regression for explanation",
            "Regression for prediction",
            "Training-validation split",
            "Regularization",
            "Statistical significance",
            "Predictive performance",
            "Real-world examples comparing explanation and prediction",
        ],
    },
    "student_persona": (
        "Grade: 2nd-year undergraduate students in business and social sciences\n"
        "Focus Time: 20 minutes\n"
        "Background Knowledge: Has learned regression for explanation in a statistics "
        "course in a business or social sciences department. Lacks ideas about "
        "training-validation split, MSE, RMSE, and regularization."
    ),
    "pedagogy": {
        "bloom_levels": ["remember", "understand"],
        "icap_level": "active",
    },
}
REQUEST_PATH.write_text(json.dumps(REQUEST, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print("Wrote", REQUEST_PATH)


Wrote /content/drive/MyDrive/education_video/requests/web_demo_regression.json


In [10]:
# 6) Smoke checks (subprocess only for manim)

from teachgen.eval.text_llm import TextLLM, _resolve_backend
from teachgen.schema import TeachingRequest

backend, base_url, _ = _resolve_backend()
print("eval backend:", backend, base_url)
assert backend == "openai"
print("topic:", TeachingRequest.model_validate_json(REQUEST_PATH.read_text(encoding="utf-8")).course.topic)

chk = subprocess.check_output(
    [sys.executable, "-c", "import manim; print(getattr(manim,'__version__','ok'))"],
    text=True,
).strip()
print("manim (subprocess):", chk)
print("smoke ok")


eval backend: openai https://api.openai.com/v1
topic: Regression for explanation vs. regression for prediction
manim (subprocess): 0.19.0
smoke ok


In [11]:
# 7) Run full evaluator loop

try:
    subprocess.check_call(
        [sys.executable, "-c", "import manim"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.STDOUT,
    )
except subprocess.CalledProcessError as e:
    raise ModuleNotFoundError(
        "manim missing in fresh process — re-run Cell 3 until DEPS OK"
    ) from e
if not shutil.which("manim"):
    raise RuntimeError("manim CLI missing — re-run Cell 3")

cmd = [
    sys.executable, "-u", "-m", "teachgen",
    "--request-json", str(REQUEST_PATH),
    "--provider", "openai",
    "--text-model", TEXT_MODEL,
    "--refinement-text-model", TEXT_MODEL,
    "--vision-model", TEXT_MODEL,
    "--plan-refinement-mode", "evaluator",
    "--feedback-mode", "evaluator",
    "--eval-baseline",
    "--max-workers", str(MAX_WORKERS),
    "--max-rounds", "3",
    "--max-plan-rounds", "1",
    "--run-dir", str(RUN_DIR),
]
print("cmd:", " ".join(cmd))
print("---")

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["PYTHONIOENCODING"] = "utf-8"
proc = subprocess.Popen(
    cmd, cwd=str(REPO), env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, encoding="utf-8", errors="replace", bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
code = proc.wait()
print("exit_code:", code)
if code != 0:
    raise SystemExit(code)


cmd: /usr/bin/python3 -u -m teachgen --request-json /content/drive/MyDrive/education_video/requests/web_demo_regression.json --provider openai --text-model gpt-5 --refinement-text-model gpt-5 --vision-model gpt-5 --plan-refinement-mode evaluator --feedback-mode evaluator --eval-baseline --max-workers 20 --max-rounds 3 --max-plan-rounds 1 --run-dir /content/drive/MyDrive/education_video/teachgen_runs
---
[teachgen] Phase 1: writing teaching content...
[teachgen] Phase 1: routing segments to renderers...
[teachgen] Phase 1: initial lesson plan -> /content/drive/MyDrive/education_video/teachgen_runs/regression-for-explanation-vs-regression-for-prediction/lesson_plan_initial.json
[teachgen] Phase 1: evaluating lesson plan (round 0)...
[teachgen]    plan_score=4.43  requires_revision=False
[teachgen] Phase 1: lesson plan -> /content/drive/MyDrive/education_video/teachgen_runs/regression-for-explanation-vs-regression-for-prediction/lesson_plan.json
[teachgen]    seg1  [concept_image] One too

: 

In [ ]:
# 8) Show Drive outputs

from IPython.display import Video, display

slug_dirs = sorted([p for p in RUN_DIR.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True)
if not slug_dirs:
    raise FileNotFoundError(f"No run dirs under {RUN_DIR}")
topic_dir = slug_dirs[0]
video = topic_dir / "video" / "final.mp4"
print("topic_dir:", topic_dir)
print("final:", video if video.exists() else "(missing)")
if video.exists():
    display(Video(str(video), embed=True, width=720))
